# 00 — Cloud Setup
Run this notebook first in Google Colab. It mounts Drive, verifies GPU, and prepares paths. Do not store secrets in the notebook.


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch, os, platform
print('Python platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## Configure project paths
Set the shared Drive root used by your team. Every member should change only this root if their mounted path differs.


In [9]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Deep Learning - RealWaste')  # CHANGE if needed
DATASET_ROOT = DRIVE_ROOT / '01_Dataset' / 'RealWaste'
SPLIT_DIR = DRIVE_ROOT / '02_Splits'
CHECKPOINT_DIR = DRIVE_ROOT / '03_Checkpoints'
RESULT_DIR = DRIVE_ROOT / '04_Results'

for p in [SPLIT_DIR, CHECKPOINT_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DATASET_ROOT exists:', DATASET_ROOT.exists())
print('SPLIT_DIR:', SPLIT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('RESULT_DIR:', RESULT_DIR)


DRIVE_ROOT: /content/drive/MyDrive/Deep Learning - RealWaste
DATASET_ROOT exists: True
SPLIT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/02_Splits
CHECKPOINT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/03_Checkpoints
RESULT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/04_Results


In [11]:
!pip -q install kagglehub

In [12]:
import kagglehub

download_path = kagglehub.dataset_download(
    "joebeachcapital/realwaste"
)

print("Downloaded to:")
print(download_path)

100%|██████████| 657M/657M [00:11<00:00, 59.7MB/s]

Extracting files...


Downloaded to:
/root/.cache/kagglehub/datasets/joebeachcapital/realwaste/versions/1


In [15]:
import shutil
from pathlib import Path

SOURCE_DATASET = download_path / "realwaste-main" / "RealWaste"
TARGET_DATASET = DATASET_ROOT

print("Source:", SOURCE_DATASET)
print("Target:", TARGET_DATASET)

assert SOURCE_DATASET.exists(), "Source dataset not found."

# Copy dataset into Drive
shutil.copytree(
    SOURCE_DATASET,
    TARGET_DATASET,
    dirs_exist_ok=True
)

print("Copy completed.")

Source: /root/.cache/kagglehub/datasets/joebeachcapital/realwaste/versions/1/realwaste-main/RealWaste
Target: /content/drive/MyDrive/Deep Learning - RealWaste/01_Dataset/RealWaste
Copy completed.


In [16]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_dirs = sorted([
    p for p in TARGET_DATASET.iterdir()
    if p.is_dir()
])

print("Classes found:", len(class_dirs))

total_images = 0

for class_dir in class_dirs:
    images = [
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print(f"{class_dir.name:25} {len(images)}")
    total_images += len(images)

print("\nTotal image files:", total_images)

Classes found: 9
Cardboard                 461
Food Organics             411
Glass                     420
Metal                     790
Miscellaneous Trash       495
Paper                     500
Plastic                   921
Textile Trash             318
Vegetation                436

Total image files: 4752


## GitHub repository
In a fresh runtime, clone the private/team repo manually or with an authenticated method approved by your team. Never paste a GitHub token into a shared notebook.
